# 01 Lunar Catcher Trade Study

Reproduces the key calculations for the lunar mass-driver -> 30 km equatorial catcher -> 0.8 km/s bridge to stable staging orbit discussion.


In [ ]:
import math
import os
import sys
from pprint import pprint

# Ensure local src helpers are importable when running this notebook from notebooks/
repo_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from frozen_retreat.src.orbital import (
    circular_speed_km_s,
    hohmann_delta_v_km_s,
    specific_energy_surface_rest_to_orbit_m2_s2,
    transfer_ellipse_from_rp_ra,
)
from frozen_retreat.src.ep import (
    ep_power_for_thrust_W,
    mass_ratio,
    thrust_for_impulse,
    thrust_from_ep_power_N,
)
from frozen_retreat.src.unit_helpers import (
    days_to_seconds,
    hours_to_seconds,
    km_s_to_m_s,
    markdown_table,
    metric_tons_to_kg,
)


In [ ]:
from frozen_retreat.src.scenario import Scenario, run_trade_study

# Scenario-driven entrypoint for notebook use
baseline_scenario = Scenario()
scenario_results = run_trade_study(baseline_scenario)
print(f"Scenario runner check: delivered_rocks_t = {scenario_results['section_8']['delivered_rocks_t']:.3f}")


## 0) Constants and Selected Values
Single source of truth for all assumptions and design values.


In [ ]:
# Constants / assumptions (units in variable names)
ASSUMPTIONS = {
    "R_moon_km": 1737.4,
    "mu_km3_s2": 4902.800,
    "h_gauntlet_km": 30.0,
    "h_escape_km": 120.0,
    "dv_bridge_m_s": 800.0,
    "ve_m_s": 3000.0,
    "eta": 0.60,
    "f_sun": 0.71,
    "t_catch_hr": 50.0,
    "m_depart_gauntlet_t": 1000.0,
    "m_dry_t": 50.0,
    "sp_solar_W_per_kg": 100.0,
    "S_W_m2": 1361.6,
    "pv_net_fraction": 0.27,
    "g0_m_s2": 9.80665,
}

ASSUMPTIONS["pv_net_W_m2"] = ASSUMPTIONS["S_W_m2"] * ASSUMPTIONS["pv_net_fraction"]
assert 300.0 < ASSUMPTIONS["pv_net_W_m2"] < 500.0

assumption_rows = [[k, v] for k, v in ASSUMPTIONS.items()]
print(markdown_table(["Parameter", "Value"], assumption_rows))
print("\nAssumptions dict:")
pprint(ASSUMPTIONS)


## 1) Basic orbital speeds at 30 km

In [ ]:
R_moon_km = ASSUMPTIONS["R_moon_km"]
mu_km3_s2 = ASSUMPTIONS["mu_km3_s2"]
h_gauntlet_km = ASSUMPTIONS["h_gauntlet_km"]

r30_km = R_moon_km + h_gauntlet_km
v_c_30_km_s = circular_speed_km_s(mu_km3_s2, r30_km)
v_c_30_m_s = km_s_to_m_s(v_c_30_km_s)

assert 1600.0 < v_c_30_m_s < 1700.0
print(f"r30_km = {r30_km:.1f} km")
print(f"v_c_30 = {v_c_30_km_s:.6f} km/s = {v_c_30_m_s:.3f} m/s")


## 2) HIGH-SCHOOL Δv: surface -> 30 km circular (two-impulse)

In [ ]:
rp_km = R_moon_km
ra_km = R_moon_km + h_gauntlet_km
xfer = transfer_ellipse_from_rp_ra(mu_km3_s2, rp_km, ra_km)

v_p_m_s = km_s_to_m_s(xfer["v_p_km_s"])
v_a_m_s = km_s_to_m_s(xfer["v_a_km_s"])
v_c_ra_m_s = km_s_to_m_s(circular_speed_km_s(mu_km3_s2, ra_km))

dv1_m_s = v_p_m_s
dv2_m_s = v_c_ra_m_s - v_a_m_s
dv_total_surface_to_30_m_s = dv1_m_s + dv2_m_s

assert 6.0 < dv2_m_s < 8.5

print(f"a = {xfer['a_km']:.3f} km")
print(f"v_p = {xfer['v_p_km_s']:.6f} km/s")
print(f"v_a = {xfer['v_a_km_s']:.6f} km/s")
print(f"v_c(ra) = {v_c_ra_m_s/1000.0:.6f} km/s")
print(f"dv1 = {dv1_m_s:.3f} m/s")
print(f"dv2 = {dv2_m_s:.3f} m/s")
print(f"dv_total = {dv_total_surface_to_30_m_s:.3f} m/s")


## 3) Relative velocity at catch at 30 km

In [ ]:
v_rel_catch_m_s = v_c_ra_m_s - v_a_m_s
assert abs(v_rel_catch_m_s - dv2_m_s) < 1e-9
print(f"v_rel_catch = {v_rel_catch_m_s:.3f} m/s")


## 4) Impact velocity if periapsis drops to the surface

In [ ]:
v_impact_m_s = v_p_m_s
print(f"v_impact = {v_impact_m_s/1000.0:.6f} km/s ({v_impact_m_s:.2f} m/s)")


## 5) Hohmann escape Δv: 30 km circular -> 120 km circular

In [ ]:
r1_km = ASSUMPTIONS["R_moon_km"] + ASSUMPTIONS["h_gauntlet_km"]
r2_km = ASSUMPTIONS["R_moon_km"] + ASSUMPTIONS["h_escape_km"]
hoh = hohmann_delta_v_km_s(ASSUMPTIONS["mu_km3_s2"], r1_km, r2_km)
dv_30_to_120_m_s = km_s_to_m_s(hoh["dv_total_km_s"])

print(f"dv1_raise = {km_s_to_m_s(hoh['dv1_km_s']):.3f} m/s")
print(f"dv2_circ  = {km_s_to_m_s(hoh['dv2_km_s']):.3f} m/s")
print(f"dv_30_to_120_total = {dv_30_to_120_m_s:.3f} m/s")


## 6) Escape power sizing (2-day and 3-day)
Catch makeup power uses delivered-rock mass from the bridge mass budget (Section 8 formula).

In [ ]:
def bridge_budget_t(m_depart_t: float, dv_bridge_m_s: float, ve_m_s: float, m_dry_t: float) -> dict:
    """Bridge mass budget in metric tons using rocket equation."""
    mr = mass_ratio(dv_bridge_m_s, ve_m_s)
    m_arrive_t = m_depart_t / mr
    delivered_rocks_t = m_arrive_t - m_dry_t
    up_prop_t = m_depart_t - m_arrive_t
    down_prop_t = (mr - 1.0) * m_dry_t
    return {
        "MR": mr,
        "m_arrive_t": m_arrive_t,
        "delivered_rocks_t": delivered_rocks_t,
        "up_prop_t": up_prop_t,
        "down_prop_t": down_prop_t,
    }

bb = bridge_budget_t(
    ASSUMPTIONS["m_depart_gauntlet_t"],
    ASSUMPTIONS["dv_bridge_m_s"],
    ASSUMPTIONS["ve_m_s"],
    ASSUMPTIONS["m_dry_t"],
)
captured_mass_kg = metric_tons_to_kg(bb["delivered_rocks_t"])

t_catch_s = hours_to_seconds(ASSUMPTIONS["t_catch_hr"])
J_catch_N_s = captured_mass_kg * v_rel_catch_m_s
T_catch_N = J_catch_N_s / t_catch_s
P_catch_W = ep_power_for_thrust_W(T_catch_N, ASSUMPTIONS["ve_m_s"], ASSUMPTIONS["eta"])

escape_case_days = [2.0, 3.0]
escape_rows = []
for days in escape_case_days:
    t_escape_s = days_to_seconds(days)
    M_loaded_kg = metric_tons_to_kg(ASSUMPTIONS["m_depart_gauntlet_t"])
    T_avg_N = thrust_for_impulse(M_loaded_kg, dv_30_to_120_m_s, t_escape_s)
    T_sun_N = T_avg_N / ASSUMPTIONS["f_sun"]
    P_sun_W = ep_power_for_thrust_W(T_sun_N, ASSUMPTIONS["ve_m_s"], ASSUMPTIONS["eta"])
    escape_rows.append({
        "escape_days": days,
        "T_avg_N": T_avg_N,
        "T_sun_N": T_sun_N,
        "P_sun_W": P_sun_W,
        "T_catch_N": T_catch_N,
        "P_catch_W": P_catch_W,
    })

print(markdown_table(
    ["escape_days", "T_avg_N", "T_sun_N", "P_sun_MW", "T_catch_N", "P_catch_kW"],
    [
        [
            f"{r['escape_days']:.0f}",
            f"{r['T_avg_N']:.2f}",
            f"{r['T_sun_N']:.2f}",
            f"{r['P_sun_W']/1e6:.3f}",
            f"{r['T_catch_N']:.2f}",
            f"{r['P_catch_W']/1e3:.3f}",
        ]
        for r in escape_rows
    ]
))


## 7) Solar array mass and area (catcher) with 100 W/kg

In [ ]:
housekeeping_margin_W = 0.03e6
system_margin_factor = 1.2
sp_W_per_kg = ASSUMPTIONS["sp_solar_W_per_kg"]
pv_net_W_m2 = ASSUMPTIONS["pv_net_W_m2"]

array_rows = []
for r in escape_rows:
    P_total_W = (r["P_sun_W"] + r["P_catch_W"] + housekeeping_margin_W) * system_margin_factor
    array_mass_kg = P_total_W / sp_W_per_kg
    array_area_m2 = P_total_W / pv_net_W_m2
    array_rows.append({
        "escape_days": r["escape_days"],
        "P_total_W": P_total_W,
        "array_mass_kg": array_mass_kg,
        "array_area_m2": array_area_m2,
        "P_sun_W": r["P_sun_W"],
    })

print(markdown_table(
    ["escape_days", "P_total_MW", "array_mass_t", "array_area_m2"],
    [
        [
            f"{r['escape_days']:.0f}",
            f"{r['P_total_W']/1e6:.3f}",
            f"{r['array_mass_kg']/1000.0:.3f}",
            f"{r['array_area_m2']:.1f}",
        ]
        for r in array_rows
    ]
))


## 8) Rocket equation for LOX-only electric at ve=3000 m/s (bridge leg)

In [ ]:
MR = bb["MR"]
m_depart_t = ASSUMPTIONS["m_depart_gauntlet_t"]
m_dry_t = ASSUMPTIONS["m_dry_t"]
m_arrive_depot_t = bb["m_arrive_t"]
delivered_rocks_t = bb["delivered_rocks_t"]
up_leg_prop_t = bb["up_prop_t"]
down_leg_prop_t = bb["down_prop_t"]

assert delivered_rocks_t > 0.0
assert up_leg_prop_t > down_leg_prop_t > 0.0

print(markdown_table(
    ["quantity", "value"],
    [
        ["MR", f"{MR:.6f}"],
        ["m_depart_gauntlet_t", f"{m_depart_t:.3f}"],
        ["m_arrive_depot_t", f"{m_arrive_depot_t:.3f}"],
        ["delivered_rocks_t", f"{delivered_rocks_t:.3f}"],
        ["up_leg_prop_t", f"{up_leg_prop_t:.3f}"],
        ["down_leg_prop_t", f"{down_leg_prop_t:.3f}"],
    ],
))


## 9) Time to climb gauntlet -> depot at sized power (2-day escape baseline)

In [ ]:
baseline_row = [r for r in array_rows if abs(r["escape_days"] - 2.0) < 1e-9][0]
P_sun_baseline_W = baseline_row["P_sun_W"]

T_sun_available_N = thrust_from_ep_power_N(P_sun_baseline_W, ASSUMPTIONS["ve_m_s"], ASSUMPTIONS["eta"])
T_avg_available_N = ASSUMPTIONS["f_sun"] * T_sun_available_N

m0_kg = metric_tons_to_kg(ASSUMPTIONS["m_depart_gauntlet_t"])
m1_kg = metric_tons_to_kg(m_arrive_depot_t)
m_avg_kg = 0.5 * (m0_kg + m1_kg)
a_avg_m_s2 = T_avg_available_N / m_avg_kg

t_up_leg_s = ASSUMPTIONS["dv_bridge_m_s"] / a_avg_m_s2
t_up_leg_days = t_up_leg_s / 86400.0

assert t_up_leg_days > 1.0
print(f"P_sun_baseline = {P_sun_baseline_W/1e6:.3f} MW")
print(f"T_sun_available = {T_sun_available_N:.3f} N")
print(f"T_avg_available = {T_avg_available_N:.3f} N")
print(f"Estimated bridge travel time = {t_up_leg_days:.2f} days")
print("Scaling: t ~ 1/P (holding ve, eta, and mission masses fixed).")


## 10) Ground mass-driver power and PV area for delivering rocks per cycle

In [ ]:
eff_driver = 0.5
pv_net_fraction_surface = ASSUMPTIONS["pv_net_fraction"]

specific_energy_m2_s2 = specific_energy_surface_rest_to_orbit_m2_s2(
    ASSUMPTIONS["mu_km3_s2"],
    ASSUMPTIONS["R_moon_km"],
    ASSUMPTIONS["R_moon_km"] + ASSUMPTIONS["h_gauntlet_km"],
)
energy_total_J = specific_energy_m2_s2 * metric_tons_to_kg(delivered_rocks_t)

power_ideal_W = energy_total_J / t_catch_s
power_elec_W = power_ideal_W / eff_driver
pv_area_surface_m2 = power_elec_W / (ASSUMPTIONS["S_W_m2"] * pv_net_fraction_surface)
pv_square_side_m = math.sqrt(pv_area_surface_m2)

print(markdown_table(
    ["quantity", "value"],
    [
        ["specific_energy_MJ_per_kg", f"{specific_energy_m2_s2/1e6:.6f}"],
        ["delivered_mass_t", f"{delivered_rocks_t:.3f}"],
        ["energy_total_TJ", f"{energy_total_J/1e12:.3f}"],
        ["power_ideal_MW", f"{power_ideal_W/1e6:.3f}"],
        ["power_elec_MW", f"{power_elec_W/1e6:.3f}"],
        ["surface_pv_area_m2", f"{pv_area_surface_m2:.1f}"],
        ["equivalent_square_side_m", f"{pv_square_side_m:.1f}"],
    ],
))


## Sensitivity Cell
Simple one-at-a-time parameter sweeps for `ve`, `eta`, `m_dry_t`, `sp_solar_W_per_kg`, and `pv_net_fraction`.

In [ ]:
def recompute_snapshot(ve_m_s, eta, m_dry_t, sp_w_per_kg, pv_frac):
    local = dict(ASSUMPTIONS)
    local["ve_m_s"] = ve_m_s
    local["eta"] = eta
    local["m_dry_t"] = m_dry_t
    local["sp_solar_W_per_kg"] = sp_w_per_kg
    local["pv_net_fraction"] = pv_frac
    local["pv_net_W_m2"] = local["S_W_m2"] * pv_frac

    bb_local = bridge_budget_t(local["m_depart_gauntlet_t"], local["dv_bridge_m_s"], ve_m_s, m_dry_t)
    captured_kg = metric_tons_to_kg(bb_local["delivered_rocks_t"])
    t_catch_local_s = hours_to_seconds(local["t_catch_hr"])
    T_catch_local = captured_kg * v_rel_catch_m_s / t_catch_local_s
    P_catch_local = ep_power_for_thrust_W(T_catch_local, ve_m_s, eta)

    t_escape_s = days_to_seconds(2.0)
    M_loaded_kg = metric_tons_to_kg(local["m_depart_gauntlet_t"])
    T_avg_local = thrust_for_impulse(M_loaded_kg, dv_30_to_120_m_s, t_escape_s)
    T_sun_local = T_avg_local / local["f_sun"]
    P_sun_local = ep_power_for_thrust_W(T_sun_local, ve_m_s, eta)

    P_total_local = (P_sun_local + P_catch_local + 0.03e6) * 1.2
    array_mass_t = (P_total_local / sp_w_per_kg) / 1000.0
    array_area_m2 = P_total_local / (local["S_W_m2"] * pv_frac)

    return {
        "MR": bb_local["MR"],
        "delivered_rocks_t": bb_local["delivered_rocks_t"],
        "P_total_MW": P_total_local / 1e6,
        "array_mass_t": array_mass_t,
        "array_area_m2": array_area_m2,
    }

sensitivity_cases = [
    ("baseline", ASSUMPTIONS["ve_m_s"], ASSUMPTIONS["eta"], ASSUMPTIONS["m_dry_t"], ASSUMPTIONS["sp_solar_W_per_kg"], ASSUMPTIONS["pv_net_fraction"]),
    ("ve+20%", ASSUMPTIONS["ve_m_s"]*1.2, ASSUMPTIONS["eta"], ASSUMPTIONS["m_dry_t"], ASSUMPTIONS["sp_solar_W_per_kg"], ASSUMPTIONS["pv_net_fraction"]),
    ("eta+10%", ASSUMPTIONS["ve_m_s"], ASSUMPTIONS["eta"]*1.1, ASSUMPTIONS["m_dry_t"], ASSUMPTIONS["sp_solar_W_per_kg"], ASSUMPTIONS["pv_net_fraction"]),
    ("dry+20t", ASSUMPTIONS["ve_m_s"], ASSUMPTIONS["eta"], ASSUMPTIONS["m_dry_t"]+20.0, ASSUMPTIONS["sp_solar_W_per_kg"], ASSUMPTIONS["pv_net_fraction"]),
    ("sp+50%", ASSUMPTIONS["ve_m_s"], ASSUMPTIONS["eta"], ASSUMPTIONS["m_dry_t"], ASSUMPTIONS["sp_solar_W_per_kg"]*1.5, ASSUMPTIONS["pv_net_fraction"]),
    ("pv_frac+0.05", ASSUMPTIONS["ve_m_s"], ASSUMPTIONS["eta"], ASSUMPTIONS["m_dry_t"], ASSUMPTIONS["sp_solar_W_per_kg"], ASSUMPTIONS["pv_net_fraction"]+0.05),
]

rows = []
for name, ve_i, eta_i, dry_i, sp_i, pv_i in sensitivity_cases:
    out = recompute_snapshot(ve_i, eta_i, dry_i, sp_i, pv_i)
    rows.append([
        name,
        f"{ve_i:.0f}",
        f"{eta_i:.3f}",
        f"{dry_i:.1f}",
        f"{sp_i:.1f}",
        f"{pv_i:.3f}",
        f"{out['delivered_rocks_t']:.2f}",
        f"{out['P_total_MW']:.3f}",
        f"{out['array_mass_t']:.3f}",
        f"{out['array_area_m2']:.1f}",
    ])

print(markdown_table(
    ["case", "ve_m_s", "eta", "dry_t", "sp_W_kg", "pv_frac", "delivered_rocks_t", "P_total_MW", "array_mass_t", "array_area_m2"],
    rows,
))


## 11) Final Consolidated Report Summary

In [ ]:
# Pull 2-day and 3-day sizing rows
row_2d = [r for r in array_rows if abs(r["escape_days"] - 2.0) < 1e-9][0]
row_3d = [r for r in array_rows if abs(r["escape_days"] - 3.0) < 1e-9][0]

report_rows = [
    ["catch_relative_velocity_m_s", f"{v_rel_catch_m_s:.3f}"],
    ["impact_velocity_km_s", f"{v_impact_m_s/1000.0:.6f}"],
    ["dv_30_to_120_m_s", f"{dv_30_to_120_m_s:.3f}"],
    ["bridge_MR", f"{MR:.6f}"],
    ["rock_delivered_per_cycle_t", f"{delivered_rocks_t:.3f}"],
    ["up_leg_prop_t", f"{up_leg_prop_t:.3f}"],
    ["down_leg_prop_t", f"{down_leg_prop_t:.3f}"],
    ["escape_2d_total_power_MW", f"{row_2d['P_total_W']/1e6:.3f}"],
    ["escape_2d_array_mass_t", f"{row_2d['array_mass_kg']/1000.0:.3f}"],
    ["escape_2d_array_area_m2", f"{row_2d['array_area_m2']:.1f}"],
    ["escape_3d_total_power_MW", f"{row_3d['P_total_W']/1e6:.3f}"],
    ["escape_3d_array_mass_t", f"{row_3d['array_mass_kg']/1000.0:.3f}"],
    ["escape_3d_array_area_m2", f"{row_3d['array_area_m2']:.1f}"],
    ["bridge_time_days_baseline", f"{t_up_leg_days:.2f}"],
    ["ground_driver_power_elec_MW", f"{power_elec_W/1e6:.3f}"],
    ["ground_driver_pv_area_m2", f"{pv_area_surface_m2:.1f}"],
]

print("Assumptions (single source of truth):")
pprint(ASSUMPTIONS)
print("\nReport summary:")
print(markdown_table(["metric", "value"], report_rows))
